# Podcast Studio: Supplements Scorecard Podcast Generation

This notebook generates a professional podcast from the supplements scorecard notes using OpenAI's LLM and TTS APIs.

In [13]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json

# Change to script directory and add src to path
notebook_dir = Path("/Users/petramifka/opt/Week 2/podcast-studio")
os.chdir(notebook_dir)
sys.path.insert(0, str(notebook_dir / 'src'))

# Load environment variables
load_dotenv()

# Import our modules
from data_processor import DataProcessor
from llm_processor import LLMProcessor
from tts_generator import TTSGenerator

print("All modules imported successfully!")
print(f"Working directory: {Path.cwd()}")

All modules imported successfully!
Working directory: /Users/petramifka/opt/Week 2/podcast-studio


## Step 1: Setup and Imports

In [19]:
# Set API key from .env file in environment before initializing
import os
from pathlib import Path

env_file = Path("/Users/petramifka/opt/Week 2/podcast-studio/.env")
if env_file.exists():
    with open(env_file, 'r') as f:
        for line in f:
            if '=' in line:
                key, value = line.strip().split('=', 1)
                os.environ[key] = value

# Initialize all processors
data_processor = DataProcessor()
llm_processor = LLMProcessor()
tts_generator = TTSGenerator()

print("Data Processor initialized")
print("LLM Processor initialized")
print("TTS Generator initialized")

Data Processor initialized
LLM Processor initialized
TTS Generator initialized


## Step 2: Initialize Processors

In [15]:
# Load the supplements file
supplements_file = "supplements_scorecard_notes.txt"

if Path(supplements_file).exists():
    content = data_processor.process_file(supplements_file)
    print(f"File loaded successfully!")
    print(f"Content length: {len(content)} characters")
    print(f"\nFirst 300 characters:\n{content[:300]}...")
else:
    print(f"Error: {supplements_file} not found!")

File loaded successfully!
Content length: 8040 characters

First 300 characters:

SUPPLEMENTS — HARVARD HEALTH ARTICLE (FORMATTED TEXT)

Dietary supplements are wildly popular. About half the adult population take at least one supplement. It's easy to understand why supplements are such big sellers. The public has a legitimate desire for good health, and the supplement industry ...


## Step 3: Load Supplements Content

In [16]:
# Validate content length
try:
    is_valid = data_processor.validate_content_length(content)
    print(f"Content validation: PASSED")
    print(f"Content is suitable for podcast generation")
except ValueError as e:
    print(f"Content validation: FAILED - {e}")

Content validation: PASSED
Content is suitable for podcast generation


## Step 4: Validate Content

In [20]:
# Generate podcast script with educational style
print("Generating podcast script...")
print("This may take a moment...\n")

script = llm_processor.generate_podcast_script(
    content=content,
    style="educational"
)

print("Podcast script generated!")
print(f"\nScript length: {len(script)} characters\n")
print("Script Preview (first 500 characters):")
print("=" * 80)
print(script[:500])
print("=" * 80)

Generating podcast script...
This may take a moment...

Podcast script generated!

Script length: 5503 characters

Script Preview (first 500 characters):
**[Podcast Intro Music Fades In]**

**Host:** Welcome back to "Health Unlocked," the podcast that digs deep into wellness topics to help you make informed choices for your health. I’m your host, [Your Name], and today, we’re diving into a hot topic that affects nearly half of the adult population in the United States: dietary supplements. 

**[Music Fades Out]**

**Host:** Have you ever wondered if those vitamins and minerals you’re taking are truly beneficial? Or, are they just an expensive way


## Step 5: Generate Podcast Script

In [31]:
# Generate audience-targeted script for 40-year-old students
print("Generating script for 40-year-old returning students...")
print("This may take a moment...\n")

# Custom prompt for target audience
targeted_prompt = """
You are a professional podcast scriptwriter creating content for 40-year-old adults returning to school.
These listeners are:
- Balancing education with work and family responsibilities
- Seeking practical, real-world health advice
- Interested in wellness that fits their busy lifestyle
- Appreciate straightforward, no-nonsense information
- May be parents and want to set health examples for their children
- Looking for evidence-based information to make informed decisions

Create a podcast script with the following structure:

**OPENING:**
Start with: "Health on the Go, navigating wellness as an Ironhacker, brought to you by APP Consulting - Artem, Petra and Pedro - and your host, the one and only Isabella."

**ANECDOTE:**
Include a funny, relatable anecdote about 40-year-olds mixing up their supplements and getting scared they might overdose. Make it humorous but realistic (like someone taking three different bottles thinking they're the same supplement, or confusing their kid's gummies with their own pills).

**CONTENT:**
Convert the following content into an engaging podcast script for THIS specific audience. Include:
- Relatable scenarios about managing health while juggling school, work, and family
- Practical tips they can implement immediately
- References to the time pressures and stress of adult education
- Emphasis on long-term health investment for themselves and their families
- Real-world cost-benefit analysis relevant to their life stage
- Encouraging tone that validates their challenges
- Examples that resonate with 40+ professionals going back to school

Keep the tone conversational, supportive, and practical.

Content to convert:
""" + content + "\n\nGenerate the podcast script now, tailored for 40-year-old returning students:"

# Use OpenAI to generate the targeted script
response = llm_processor.client.chat.completions.create(
    model=llm_processor.model,
    messages=[{"role": "user", "content": targeted_prompt}],
    temperature=0.7,
    max_tokens=2500
)

targeted_script = response.choices[0].message.content

print("Targeted podcast script generated!")
print(f"\nScript length: {len(targeted_script)} characters\n")
print("Script Preview (first 600 characters):")
print("=" * 80)
print(targeted_script[:600])
print("=" * 80)

Generating script for 40-year-old returning students...
This may take a moment...

Targeted podcast script generated!

Script length: 5214 characters

Script Preview (first 600 characters):
**OPENING:**

[Upbeat music fades in]

Isabella: “Health on the Go, navigating wellness as an Ironhacker, brought to you by APP Consulting - Artem, Petra, and Pedro - and your host, the one and only Isabella. Welcome back, everyone! Today we’re diving into a topic that’s as confusing as your kid's math homework and as essential as a good cup of coffee on a Monday morning: supplements! 

But first, let’s kick things off with a little story. 

**ANECDOTE:**

Isabella: So, the other day, I was rushing out the door, balancing my laptop bag, a lunchbox for the kids, and—wait for it—a handful of sup


In [34]:
# Create metadata for the targeted podcast
targeted_metadata = {
    "title": "Supplements Scorecard: For Busy 40+ Students",
    "subtitle": "Evidence-Based Wellness for Your Next Chapter",
    "source": "Harvard Health Article on Dietary Supplements",
    "target_audience": "40-year-old adults returning to school",
    "audience_characteristics": [
        "Balancing education, work, and family",
        "Seeking practical health advice",
        "Interested in time-efficient wellness",
        "Evidence-based decision makers",
        "Parents modeling healthy behaviors"
    ],
    "style": "conversational, supportive, practical",
    "voice": "nova",
    "content_length": len(content),
    "script_length": len(targeted_script),
    "audio_files": targeted_audio_files,
    "num_chunks": len(targeted_audio_files),
    "created_date": "February 9, 2026"
}

# Save metadata
targeted_metadata_file = output_dir / "podcast_metadata_40yo_students.json"
with open(targeted_metadata_file, 'w') as f:
    json.dump(targeted_metadata, f, indent=2)

print(f"Targeted podcast metadata saved to: {targeted_metadata_file}")
print("\nTargeted Podcast Summary:")
print("=" * 80)
print(f"Title: {targeted_metadata['title']}")
print(f"Subtitle: {targeted_metadata['subtitle']}")
print(f"Target Audience: {targeted_metadata['target_audience']}")
print(f"Style: {targeted_metadata['style']}")
print(f"Voice: {targeted_metadata['voice']}")
print(f"Audio files generated: {len(targeted_audio_files)}")
print(f"Script length: {targeted_metadata['script_length']} characters")
print("=" * 80)

Targeted podcast metadata saved to: podcast_output/podcast_metadata_40yo_students.json

Targeted Podcast Summary:
Title: Supplements Scorecard: For Busy 40+ Students
Subtitle: Evidence-Based Wellness for Your Next Chapter
Target Audience: 40-year-old adults returning to school
Style: conversational, supportive, practical
Voice: nova
Audio files generated: 2
Script length: 5214 characters


## Step 5B: Generate Audience-Targeted Script (40-Year-Old Students)

In [33]:
# Generate audio from the targeted script
print("Generating audio for targeted script...")
print("This will split the script into chunks and generate audio for each chunk.\n")

# Create a separate directory for targeted audio
targeted_audio_dir = Path("audio_output/targeted_40yo")
targeted_audio_dir.mkdir(parents=True, exist_ok=True)

# Save audio files with targeted naming
targeted_audio_files = []
chunks = tts_generator._split_text(targeted_script, chunk_size=4000)

for i, chunk in enumerate(chunks, 1):
    output_file = f"supplements_podcast_40yo_students_part_{i:03d}.mp3"
    file_path = targeted_audio_dir / output_file
    
    response = tts_generator.client.audio.speech.create(
        model="tts-1",
        voice=tts_generator.voice,
        input=chunk,
        speed=tts_generator.speed
    )
    
    response.stream_to_file(str(file_path))
    targeted_audio_files.append(str(file_path))
    print(f"Generated: {output_file}")

print(f"\nGenerated {len(targeted_audio_files)} audio file(s) for 40-year-old students")

Generating audio for targeted script...
This will split the script into chunks and generate audio for each chunk.



/var/folders/jm/btjwvh21639_zyqy4jpnbm400000gn/T/ipykernel_68394/3965527041.py:24: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(str(file_path))


Generated: supplements_podcast_40yo_students_part_001.mp3
Generated: supplements_podcast_40yo_students_part_002.mp3

Generated 2 audio file(s) for 40-year-old students


## Step 5D: Generate Audio for Targeted Script

In [32]:
# Save the targeted script
targeted_script_file = output_dir / "supplements_podcast_script_40yo_students.txt"
with open(targeted_script_file, 'w') as f:
    f.write(targeted_script)

print(f"Targeted script saved to: {targeted_script_file}")

Targeted script saved to: podcast_output/supplements_podcast_script_40yo_students.txt


## Step 5C: Save Targeted Script

In [21]:
# Save the script to a file
output_dir = Path("podcast_output")
output_dir.mkdir(exist_ok=True)

script_file = output_dir / "supplements_podcast_script.txt"
with open(script_file, 'w') as f:
    f.write(script)

print(f"Script saved to: {script_file}")

Script saved to: podcast_output/supplements_podcast_script.txt


## Step 6: Save Podcast Script

In [22]:
# Extract key points from the content
print("Extracting key points...\n")

key_points = llm_processor.extract_key_points(content)

print("Key Points:")
print("=" * 80)
for i, point in enumerate(key_points, 1):
    print(f"{i}. {point}")
print("=" * 80)

Extracting key points...

Key Points:
1. ```json
2. [
3.     "About half of adults take dietary supplements, but they are not regulated by the FDA like medications.",
4.     "The Dietary Supplement Health and Education Act limits the FDA's ability to ensure the safety and efficacy of dietary supplements.",
5.     "Many popular supplements, such as antioxidants and multivitamins, have not been proven effective in preventing illness.",
6.     "Vitamin D is important for bone health, and supplements are recommended for those at risk of deficiency.",
7.     "Vitamin B12 supplements may be necessary for strict vegetarians and older adults, while folate is crucial for women who are pregnant or trying to conceive.",
8.     "Fish oil supplements may benefit those with cardiovascular disease who do not consume fish regularly.",
9.     "Consumers should be cautious of extravagant claims, testimonials, and potential supplement-drug interactions."
10. ]
11. ```


## Step 7: Extract Key Points

In [23]:
# Generate audio from the script
print("Generating audio files from podcast script...")
print("This will split the script into chunks and generate audio for each chunk.\n")

audio_files = tts_generator.generate_audio_chunks(
    text=script,
    chunk_size=4000,
    output_prefix="supplements_podcast"
)

print(f"\nGenerated {len(audio_files)} audio file(s):")
for i, audio_file in enumerate(audio_files, 1):
    print(f"  {i}. {audio_file}")

Generating audio files from podcast script...
This will split the script into chunks and generate audio for each chunk.


Generated 2 audio file(s):
  1. audio_output/supplements_podcast_part_001.mp3
  2. audio_output/supplements_podcast_part_002.mp3


## Step 8: Generate Podcast Audio

In [24]:
# Merge audio files into a single podcast
if len(audio_files) > 1:
    print("Merging audio files into a single podcast...\n")
    try:
        merged_audio = tts_generator.merge_audio_files(
            audio_files=audio_files,
            output_file="supplements_podcast_final.mp3"
        )
        print(f"Successfully merged audio!")
        print(f"Final podcast: {merged_audio}")
    except Exception as e:
        print(f"Note: Audio merging requires ffmpeg installed.")
        print(f"Individual audio files are available above.")
        print(f"Error: {e}")
else:
    print("Only one audio file generated. No merging needed.")
    print(f"Podcast ready: {audio_files[0]}")

Merging audio files into a single podcast...

Note: Audio merging requires ffmpeg installed.
Individual audio files are available above.
Error: [Errno 2] No such file or directory: 'ffmpeg'


## Step 9: Merge Audio Files

In [25]:
# Create metadata file for the podcast
metadata = {
    "title": "Supplements Scorecard: Harvard Health Educational Podcast",
    "source": "Harvard Health Article on Dietary Supplements",
    "style": "educational",
    "voice": "nova",
    "content_length": len(content),
    "script_length": len(script),
    "audio_files": audio_files,
    "key_points": key_points,
    "created_date": str(Path.cwd())
}

metadata_file = output_dir / "podcast_metadata.json"
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Podcast metadata saved to: {metadata_file}")
print("\nPodcast Summary:")
print("=" * 80)
print(f"Title: {metadata['title']}")
print(f"Style: {metadata['style']}")
print(f"Voice: {metadata['voice']}")
print(f"Audio files generated: {len(audio_files)}")
print(f"Original content: {metadata['content_length']} characters")
print(f"Script generated: {metadata['script_length']} characters")
print("=" * 80)

Podcast metadata saved to: podcast_output/podcast_metadata.json

Podcast Summary:
Title: Supplements Scorecard: Harvard Health Educational Podcast
Style: educational
Voice: nova
Audio files generated: 2
Original content: 8040 characters
Script generated: 5503 characters


## Step 10: Generate Podcast Metadata

In [26]:
# Display all output files
print("Podcast Generation Complete!")
print("\nOutput Files:")
print("=" * 80)
for item in output_dir.glob('*'):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        print(f"  {item.name} ({size_kb:.1f} KB)")
print("=" * 80)
print(f"\nAll files saved to: {output_dir.absolute()}")

Podcast Generation Complete!

Output Files:
  podcast_metadata.json (1.4 KB)
  supplements_podcast_script.txt (5.4 KB)

All files saved to: /Users/petramifka/opt/Week 2/podcast-studio/podcast_output


## Step 11: Summary and Output Files